### 1. Preprocesarea datelor și scalarea (Min-Max, Normalizare, Log)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

df_base = housing.copy()
drop_cols = [c for c in ['SalePrice', 'Order', 'PID'] if c in df_base.columns]

num_cols = [c for c in df_base.select_dtypes(include=np.number).columns if c not in drop_cols]
cat_cols = [c for c in df_base.select_dtypes(exclude=np.number).columns if c not in drop_cols]

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_cat = pd.DataFrame(encoder.fit_transform(df_base[cat_cols]), columns=cat_cols, index=df_base.index)
X_num = df_base[num_cols].copy()
X_raw = pd.concat([X_num, X_cat], axis=1)

minmax_scaler = MinMaxScaler()
X_minmax = pd.DataFrame(minmax_scaler.fit_transform(X_raw), columns=X_raw.columns, index=X_raw.index)

std_scaler = StandardScaler()
X_std = pd.DataFrame(std_scaler.fit_transform(X_raw), columns=X_raw.columns, index=X_raw.index)

normalizer = Normalizer()
X_norm = pd.DataFrame(normalizer.fit_transform(X_minmax), columns=X_raw.columns, index=X_raw.index)

X_log = X_raw.copy()
for col in num_cols:
    if (X_log[col] >= 0).all() and X_log[col].skew() > 1.0:
        X_log[col] = np.log1p(X_log[col])
X_log_scaled = pd.DataFrame(MinMaxScaler().fit_transform(X_log), columns=X_log.columns, index=X_log.index)

### 2. PCA și pragurile de varianță

In [ ]:
pca_full = PCA(random_state=42)
pca_full.fit(X_minmax)

cum_var = np.cumsum(pca_full.explained_variance_ratio_)
thresholds = [0.98, 0.95, 0.90, 0.80]
n_comp_by_thresh = {}

for t in thresholds:
    n_comp = int(np.argmax(cum_var >= t) + 1)
    n_comp_by_thresh[t] = n_comp
    print(f"Prag {int(t*100)}%: {n_comp} componente principale (varianță cumulată = {cum_var[n_comp-1]:.4f})")

for i in range(min(10, len(pca_full.explained_variance_ratio_))):
    print(f"PC{i+1}: {pca_full.explained_variance_ratio_[i]:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(cum_var) + 1), cum_var, marker='o', markersize=3)
for t in thresholds:
    n_c = n_comp_by_thresh[t]
    plt.axhline(y=t, linestyle='--', alpha=0.6, label=f"{int(t*100)}% -> {n_c} PCs")
    plt.scatter([n_c], [cum_var[n_c - 1]], zorder=5)
plt.xlabel("Număr de componente principale")
plt.ylabel("Varianță cumulată explicată")
plt.title("Varianța cumulată în funcție de numărul de componente principale")
plt.legend()
plt.grid(True)
plt.show()

### 3. K-Means Clustering pe pragurile de varianță

In [ ]:
k_values = range(2, 9)
pca_models = {}
X_pca_sets = {}

for t in thresholds:
    pca_t = PCA(n_components=t, random_state=42)
    X_t = pca_t.fit_transform(X_minmax)
    pca_models[t] = pca_t
    X_pca_sets[t] = X_t

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, t in enumerate(thresholds):
    X_t = X_pca_sets[t]
    inertias = []
    sil_scores = []
    for k in k_values:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels_k = km.fit_predict(X_t)
        inertias.append(km.inertia_)
        sil_scores.append(silhouette_score(X_t, labels_k))
    ax1 = axes[idx]
    ax1.plot(list(k_values), inertias, marker='o', color='tab:blue', label='Inertia')
    ax1.set_xlabel("Număr de clustere (k)")
    ax1.set_ylabel("Inertia", color='tab:blue')
    ax1.set_title(f"Elbow & Silhouette — Prag varianță {int(t*100)}% ({X_t.shape[1]} PCs)")
    ax2 = ax1.twinx()
    ax2.plot(list(k_values), sil_scores, marker='s', color='tab:orange', label='Silhouette')
    ax2.set_ylabel("Silhouette Score", color='tab:orange')

plt.tight_layout()
plt.show()

best_k = 3
X_98 = X_pca_sets[0.98]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for idx, k in enumerate([2, 3, 4]):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_98)
    s_vals = silhouette_samples(X_98, lbl)
    s_avg = silhouette_score(X_98, lbl)
    y_lower = 10
    ax = axes[idx]
    for c in range(k):
        c_vals = np.sort(s_vals[lbl == c])
        y_upper = y_lower + len(c_vals)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_vals, alpha=0.7)
        y_lower = y_upper + 10
    ax.axvline(x=s_avg, color="red", linestyle="--")
    ax.set_title(f"Silhouette k={k} (avg={s_avg:.3f})")
    ax.set_xlabel("Coeficient Silhouette")
    ax.set_ylabel("Clustere / eșantioane")

plt.tight_layout()
plt.show()

### 4. Vizualizare 2D (PC1 vs PC2) cu centroizi și comparația preprocesărilor

In [ ]:
km_98 = KMeans(n_clusters=best_k, random_state=42, n_init=10)
clusters_98 = km_98.fit_predict(X_98)
centroids_98 = km_98.cluster_centers_

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_98[:, 0], X_98[:, 1], c=clusters_98, cmap='tab10', alpha=0.6, s=25, label='Observații')
plt.scatter(centroids_98[:, 0], centroids_98[:, 1], c='black', marker='X', s=220, linewidths=2, label='Centroizi')
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"K-Means (k={best_k}) pe primele 2 componente principale (PCA 98%)")
plt.legend(*scatter.legend_elements(), title="Cluster", loc="upper right")
plt.grid(True)
plt.show()

variants = {
    "MinMax": X_minmax,
    "StandardScaler": X_std,
    "MinMax+Normalizer": X_norm,
    "Log1p+MinMax": X_log_scaled
}

for name, X_var in variants.items():
    pca_v = PCA(n_components=0.98, random_state=42)
    X_v_pca = pca_v.fit_transform(X_var)
    km_v = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    lbl_v = km_v.fit_predict(X_v_pca)
    print(f"{name:18s} | PCs: {X_v_pca.shape[1]:2d} | Silhouette: {silhouette_score(X_v_pca, lbl_v):.4f} | Inertia: {km_v.inertia_:.2f}")

### 5. PCA loadings, profilul clusterelor și predicție pe case noi

In [ ]:
pca_best = pca_models[0.98]
loadings = pd.DataFrame(
    pca_best.components_.T * np.sqrt(pca_best.explained_variance_),
    index=X_minmax.columns,
    columns=[f"PC{i+1}" for i in range(pca_best.n_components_)]
)

top_pc1 = loadings["PC1"].abs().sort_values(ascending=False).head(10).index
top_pc2 = loadings["PC2"].abs().sort_values(ascending=False).head(10).index
top_feats = list(dict.fromkeys(list(top_pc1) + list(top_pc2)))

plt.figure(figsize=(10, 6))
sns.heatmap(loadings.loc[top_feats, ["PC1", "PC2", "PC3"]], annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Loadings pe primele componente principale (PC1 - PC3)")
plt.show()

df_profile = df_base.copy()
df_profile["Cluster"] = clusters_98
summary_cols = [c for c in ["SalePrice", "OverallQual", "GrLivArea", "TotalBsmtSF", "YearBuilt", "GarageArea"] if c in df_profile.columns]
display(df_profile.groupby("Cluster")[summary_cols].agg(["mean", "median", "count"]))

new_houses_dict = {
    col: [df_base[col].median() if col in num_cols else "None" for _ in range(3)]
    for col in X_raw.columns
}
new_houses = pd.DataFrame(new_houses_dict)

if "OverallQual" in new_houses.columns:
    new_houses["OverallQual"] = [4, 7, 9]
if "GrLivArea" in new_houses.columns:
    new_houses["GrLivArea"] = [900, 1650, 2800]
if "YearBuilt" in new_houses.columns:
    new_houses["YearBuilt"] = [1955, 2002, 2018]
if "GarageArea" in new_houses.columns:
    new_houses["GarageArea"] = [0, 480, 820]

new_cat = pd.DataFrame(encoder.transform(new_houses[cat_cols]), columns=cat_cols)
new_combined = pd.concat([new_houses[num_cols], new_cat], axis=1)
new_scaled = minmax_scaler.transform(new_combined)
new_pca = pca_best.transform(new_scaled)
pred_clusters = km_98.predict(new_pca)

for i, c in enumerate(pred_clusters):
    print(f"Casa nouă #{i+1} -> Cluster {c}")

### 6.Scatter 3D

In [ ]:
pca_3d = PCA(n_components=3, random_state=42)
X_3d = pca_3d.fit_transform(X_minmax)
km_3d = KMeans(n_clusters=best_k, random_state=42, n_init=10)
lbl_3d = km_3d.fit_predict(X_3d)

df_3d = pd.DataFrame(X_3d, columns=["PC1", "PC2", "PC3"], index=df_base.index)
df_3d["Cluster"] = lbl_3d.astype(str)

bins = [0, 100000, 130000, 180000, 250000, 400000, np.inf]
price_labels = ["<100K", "100K-130K", "130K-180K", "180K-250K", "250K-400K", ">=400K"]
df_3d["PriceClassL2"] = pd.cut(df_base["SalePrice"], bins=bins, labels=price_labels, right=False).astype(str)

fig_3d_clusters = px.scatter_3d(
    df_3d, x="PC1", y="PC2", z="PC3", color="Cluster",
    title=f"3D Scatter PCA (k={best_k} Clustere) + Centroizi",
    opacity=0.65
)
fig_3d_clusters.add_trace(go.Scatter3d(
    x=km_3d.cluster_centers_[:, 0],
    y=km_3d.cluster_centers_[:, 1],
    z=km_3d.cluster_centers_[:, 2],
    mode="markers",
    marker=dict(size=10, color="black", symbol="diamond"),
    name="Centroizi"
))
fig_3d_clusters.show()

fig_3d_price = px.scatter_3d(
    df_3d, x="PC1", y="PC2", z="PC3", color="PriceClassL2",
    title="3D Scatter PCA colorat după cele 6 clase de preț (L2)",
    opacity=0.65
)
fig_3d_price.show()

display(pd.crosstab(df_3d["Cluster"], df_3d["PriceClassL2"]))